In [5]:
import os
from phoenix.otel import register

# Add Phoenix API Key for tracing
PHOENIX_API_KEY = os.getenv("PHOENIX_API_KEY")
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={PHOENIX_API_KEY}"

# configure the Phoenix tracer
tracer_provider = register(
  endpoint="https://app.phoenix.arize.com/v1/traces",
) 

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/v1/traces
|  Transport: HTTP
|  Transport Headers: {'api_key': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [6]:
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [9]:
from pydantic_ai import Agent

agent = Agent(  
    'openai:gpt-4o',
    system_prompt='Be concise, reply with one sentence.',  
)

result = await agent.run('How many feet are in a mile?')  
print(result.data)

Failed to export batch code: 204, reason: 


There are 5,280 feet in a mile.


In [10]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
from dataclasses import dataclass

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext


@dataclass
class NameContext:
    """Dependencies (context) for the conversation."""
    user_name: str


class GreetingResult(BaseModel):
    """Structured output from the AI."""
    greeting: str = Field(description="A short greeting to the user")


greeting_agent = Agent(
    model="openai:gpt-4o",
    deps_type=NameContext,
    result_type=GreetingResult,
    system_prompt=(
        "You are a personalized greeter AI. "
        "Return a short greeting for the user."
    ),
)


@greeting_agent.system_prompt
async def add_user_name(ctx: RunContext[NameContext]) -> str:
    return f"The user's name is {ctx.deps.user_name!r}."


async def main():
    deps = NameContext(user_name="Alice")

    result = await greeting_agent.run(
        "Hi, can you greet me?",
        deps=deps
    )

    print(result.data)

await main()

Failed to export batch code: 204, reason: 


greeting="Hello, Alice! It's great to see you!"


```
{"messages": [{"role": "system", "content": "You are a personalized greeter AI. Return a short greeting for the user."}, {"role": "system", "content": "The user's name is 'Alice'."}, {"role": "user", "content": "Hi, can you greet me?"}], "model": "gpt-4o", "n": 1, "parallel_tool_calls": true, "stream": false, "tool_choice": "required", "tools": [{"type": "function", "function": {"name": "final_result", "description": "Structured output from the AI.", "parameters": {"properties": {"greeting": {"description": "A short greeting to the user", "title": "Greeting", "type": "string"}}, "required": ["greeting"], "title": "GreetingResult", "type": "object"}}}]}
```